<a href="https://colab.research.google.com/github/heberdavi/mba-engsoft-tcc/blob/hybrid-rag-poc/notebooks/01-processamento_pln.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

📑 Guia de Execução Estratégica
⚠️ IMPORTANTE: Sempre que o Runtime (Ambiente de Execução) for reiniciado, as Células 1 e 2 devem ser executadas obrigatoriamente para restabelecer os caminhos do Drive e reinstalar as bibliotecas.

🔄 Fluxo de Dependências:
Sessão Recém-Iniciada: Executar Célula 1 ➔ Célula 2.

Primeira vez no projeto: Executar Célula 1 ➔ Célula 2 ➔ Célula 3 (Carga).

Retomando Processamento: Se o banco já existe no Drive, pule a Célula 3 e vá direto para a Célula 4 e/ou 5 e/ou 6.

In [ ]:
# Célula 1: Montagem do Google Drive e Configuração de Caminhos
from google.colab import drive
import os

# 1. Montagem Segura: Só executa se ainda não estiver montado
if not os.path.exists('/content/drive/MyDrive'):
    print("📂 Montando Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive já está montado e acessível.")

# 2. Configuração Estrita de Caminhos
DRIVE_DIR = '/content/drive/MyDrive/pln/hybrid-rag-poc'
DB_FILE_NAME = 'data/base-dados.db'
DB_PATH = os.path.join(DRIVE_DIR, DB_FILE_NAME)

# Artefatos SQL
SCHEMA_SQL = os.path.join(DRIVE_DIR, 'sql/01-schema.sql')
SEED_SQL = os.path.join(DRIVE_DIR, 'sql/02-seed_data.sql')

# Pasta de Saída (Outputs)
EXPORT_PATH = os.path.join(DRIVE_DIR, 'outputs')
if not os.path.exists(EXPORT_PATH):
    os.makedirs(EXPORT_PATH)
    print(f"📁 Pasta de exportação criada em: {EXPORT_PATH}")

print(f"📍 Banco de Dados: {DB_PATH}")

In [ ]:
# Célula 2: Instalação das bibliotecas e inicialização da estrutura (Schema)

# 1. Instalação Silenciosa
!pip install -q transformers torch pandas bertopic pysentimiento spacy
!python -m spacy download pt_core_news_lg -q

import sqlite3
import torch

# 2. Hardware Check para BERTimbau/BERTopic
device = 0 if torch.cuda.is_available() else -1

def inicializar_estrutura_db(db_path, schema_path):
    """Garante que a estrutura de tabelas esteja presente."""
    print(f"🛠️ Verificando integridade das tabelas...")

    # Se o arquivo de banco não existir, o SQLite o criará automaticamente
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        with open(schema_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())
        conn.commit()
        print("✅ Estrutura (Schema) validada com sucesso!")
    except Exception as e:
        print(f"❌ Erro ao processar Schema: {e}")
    finally:
        conn.close()

# 3. Execução
inicializar_estrutura_db(DB_PATH, SCHEMA_SQL)

print(f"\n🚀 Ambiente pronto (GPU: {'Ativa' if device == 0 else 'Inativa'}).")

In [ ]:
# Célula 3: Carga Inicial de Dados (Seed SQL)
def executar_carga_dados(db_path, seed_path):
    """Popula o banco apenas se a tabela 'verso' estiver vazia."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        # Verifica se já existem dados para evitar duplicidade no Drive
        cursor.execute("SELECT count(*) FROM verso")
        total_existente = cursor.fetchone()[0]

        if total_existente > 0:
            print(f"ℹ️ O banco já contém {total_existente} versos. Carga inicial ignorada.")
            return

        print("🌱 Semeando dados iniciais (02-seed_data.sql)... Isso pode levar alguns minutos.")
        with open(seed_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())

        conn.commit()
        print(f"✅ Carga de {seed_path} concluída com sucesso!")

    except sqlite3.OperationalError as e:
        print(f"⚠️ Erro operacional: {e}. Verifique se a Célula 2 foi executada.")
    except Exception as e:
        print(f"❌ Erro crítico na carga: {e}")
    finally:
        conn.close()

# Executa a carga (Somente se necessário)
executar_carga_dados(DB_PATH, SEED_SQL)

In [ ]:
# Célula 4: Limpeza Estrutural e Filtro de Densidade (Antidoto ao Ruído Nominal)
import spacy
import sqlite3
import pandas as pd
import re

# Carrega o modelo de português
try:
    nlp = spacy.load("pt_core_news_lg")
except:
    !python -m spacy download pt_core_news_lg
    nlp = spacy.load("pt_core_news_lg")

def limpar_texto_estrutural(texto):
    if not texto or len(texto.strip()) < 3: return "RUIDO_CURTO"

    doc = nlp(texto)

    # Filtramos tokens válidos (substantivos, verbos, adjetivos e nomes próprios)
    # Ignoramos stop words e pontuação
    tokens = [t for t in doc if not t.is_stop and not t.is_punct and t.pos_ in ['NOUN', 'VERB', 'ADJ', 'PROPN']]

    if not tokens: return "RUIDO_VAZIO"

    # Métrica 1: Densidade de Nomes Próprios (PROPN)
    propn_count = len([t for t in tokens if t.pos_ == 'PROPN'])
    propn_ratio = propn_count / len(tokens)

    # Métrica 2: Presença de Ação/Estado
    has_action_or_state = any(t.pos_ in ['VERB', 'ADJ'] for t in tokens)

    # CASO CRÍTICO (Ex: Nesias e Hatifa):
    if len(tokens) <= 3 and propn_ratio > 0.5 and not has_action_or_state:
        return "RUIDO_NOMINAL"

    # --- NOVO: FILTRO DINÂMICO PARA TEXTOS CURTOS ---
    # Se a frase original resultar em menos de 5 tokens significativos,
    # a poda agressiva destruiria o sentido existencial.
    # Nesse caso, retornamos o texto original inteiro, removendo apenas a pontuação.
    if len(tokens) < 5:
        return " ".join([t.text.lower() for t in doc if not t.is_punct])

    # Caso padrão: retornamos o texto limpo (apenas a essência)
    return " ".join([t.text.lower() for t in tokens])

# Execução e Persistência
conn = sqlite3.connect(DB_PATH)
df_versos = pd.read_sql_query("SELECT id, texto FROM verso WHERE livro_id = 18", conn)

print("🧼 Limpando textos e aplicando filtros de densidade gramatical...")
df_versos['texto_limpo'] = df_versos['texto'].apply(limpar_texto_estrutural)

cursor = conn.cursor()
# --- NOVO: DELEÇÃO SELETIVA ---
# Apaga apenas os registros de Jó na tabela verso_limpo antes de inserir os novos, protegendo o resto do banco.
cursor.execute("DELETE FROM verso_limpo WHERE verso_id IN (SELECT id FROM verso WHERE livro_id = 18)")

df_versos[['id', 'texto_limpo']].rename(columns={'id': 'verso_id'}).to_sql(
    'verso_limpo',
    conn,
    if_exists='append',
    index=False
)

conn.commit()
conn.close()
print("✅ Célula 4 concluída! Ruídos nominais e estruturais pré-identificados.")

In [ ]:
# Célula 5: Classificação por Eixos Existenciais (Versão Jó/PoC com Janelamento no Texto Original)
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from transformers import pipeline
import sqlite3
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# 1. Configuração e Carga de Dados (Restrito ao livro de Jó)
conn = sqlite3.connect(DB_PATH)
query = """
    SELECT v.id as verso_id, v.texto, l.abreviacao, g.id as genero_id, vl.texto_limpo
    FROM verso v
    JOIN verso_limpo vl ON v.id = vl.verso_id
    JOIN livro l ON l.id = v.livro_id
    JOIN genero_literario g ON g.id = l.genero_id
    WHERE v.livro_id = 18
    ORDER BY v.id
"""
df_input = pd.read_sql_query(query, conn)

# Tratamento de erro de tipo
df_input['texto'] = df_input['texto'].fillna('').astype(str)
df_input['texto_limpo'] = df_input['texto_limpo'].fillna('vazio').astype(str)

# --- NOVO: JANELAMENTO USANDO O TEXTO ORIGINAL ---
# O texto original possui a sintaxe completa (conectivos, preposições, etc.),
# o que é essencial para o mecanismo de atenção do BERTimbau.
df_input['texto_anterior'] = df_input['texto'].shift(1, fill_value='')
df_input['texto_posterior'] = df_input['texto'].shift(-1, fill_value='')
df_input['texto_janela'] = (df_input['texto_anterior'] + " " +
                            df_input['texto'] + " " +
                            df_input['texto_posterior']).str.strip()

# Prepara os documentos: se a Célula 4 marcou como ruído, passamos "vazio"
# para economizar processamento do Transformer.
docs_para_classificar = []
for idx, row in df_input.iterrows():
    if row['texto_limpo'].startswith('RUIDO'):
        docs_para_classificar.append("vazio")
    else:
        docs_para_classificar.append(row['texto_janela'])

# 2. Inicialização do Modelo BERTimbau (Zero-Shot)
embedding_model = pipeline("feature-extraction", model="neuralmind/bert-base-portuguese-cased", device=0)

# Sementes Metafóricas de Jó
descricoes_eixos = [
    "Estados de Fadiga Existencial e Alívio da Alma: Descreve o peso da existência, o estar psicologicamente sobrecarregado, o desânimo e o abatimento espiritual. Inclui sentimentos de angústia profunda, o clamor por socorro, dor no corpo, choro nas cinzas, feridas, gemidos, tempestade, o abismo, e a incapacidade física ou mental. Em contrapartida, oferece o fortalecimento do exausto, o pastor que guia a ovelha, o bálsamo, a renovação das energias, o refúgio na aflição, o refrigério da paz interior e o descanso para a alma atribulada.",
    "Fragilidade Humana e Firmeza Espiritual: Aborda a brevidade da vida, a impermanência dos dias, a sombra que passa, o pó, o vento que sopra, e a insegurança das coisas mundanas que perecem como a erva. Em oposição, destaca a Rocha Eterna, a fortaleza, o escudo, o fundamento inabalável, a confiança irremovível em Deus e a segurança de quem constrói a identidade sobre valores eternos e constantes.",
    "Crise de Sentido e Vocação Existencial: O sentimento de que tudo é vaidade, ilusão, neblina, e a futilidade de uma vida sem significado. Trata do vácuo existencial, do andar em trevas e da desorientação mental. Contrastando com isso, apresenta o oleiro e o barro, a luz no caminho, o ser chamado por Deus, a eleição divina, a descoberta de um sentido para a vida, a vocação, a esperança futura e a compreensão de uma missão que transcende o material.",
    "Registros Narrativos, Leis e Informações Factuais: Conteúdo puramente informativo, administrativo ou instrutivo. Inclui genealogias, listas de nomes, rituais, censos e relatos de viagens. Abrange fundamentalmente fórmulas de introdução de diálogos e marcadores de transição narrativa como 'disse', 'respondeu', 'falou', 'perguntou', servindo como uma categoria técnica para textos descritivos."
]

model_topic = BERTopic(
    embedding_model=embedding_model,
    zeroshot_topic_list=descricoes_eixos,
    zeroshot_min_similarity=0.1,
    calculate_probabilities=True,
    vectorizer_model=CountVectorizer(ngram_range=(1, 2))
)

print("🤖 Classificando via Zero-Shot com contexto original ampliado...")
topics, probs_matrix = model_topic.fit_transform(docs_para_classificar)

# 3. Processamento de Decisões e Preparação do DataFrame Final
rows_to_persist = []
print("⚖️ Processando métricas e aplicando thresholds dinâmicos...")

for i, row in tqdm(df_input.iterrows(), total=len(df_input), desc="Processando Versículos", unit="v"):

    # Se foi identificado como ruído na Célula 4, ignora o Transformer e força Eixo Narrativo
    if row['texto_limpo'].startswith('RUIDO'):
        decisao_id = 3
        status = row['texto_limpo'] # Registra qual foi o tipo de ruído (ex: RUIDO_NOMINAL)
        p_ex, p_tr, p_va, p_na = 0.0, 0.0, 0.0, 1.0
        best_score = 1.0
        margem = 1.0
        entropia = 0.0
        gap = 1.0
    else:
        p_ex = probs_matrix[i][0]
        p_tr = probs_matrix[i][1]
        p_va = probs_matrix[i][2]
        p_na = probs_matrix[i][3]

        todas_probs = sorted([p_ex, p_tr, p_va, p_na], reverse=True)
        gap = todas_probs[0] - todas_probs[1]
        entropia = -sum([p * np.log(p + 1e-9) for p in [p_ex, p_tr, p_va, p_na]])

        existenciais = [p_ex, p_tr, p_va]
        best_idx = np.argmax(existenciais)
        best_score = existenciais[best_idx]

        margem = best_score - p_na
        gen_id = row['genero_id']
        t_len = len(row['texto'])

        decisao_id = 3 # Default: Narrativo
        status = "Descarte"

        # --- Lógica Híbrida de Decisão ---
        if t_len < 35 and margem < 0.25:
            decisao_id = 3
            status = "Filtro Brevidade"
        else:
            if gen_id in [1, 2]: # Pentateuco/Histórico
                if best_score > 0.88 and margem > 0.15:
                    decisao_id, status = best_idx, "Rigor Máximo"
            elif gen_id in [3, 4]: # Poético/Profético
                if best_score > 0.50:
                    decisao_id, status = best_idx, "Sensibilidade Poética"
            elif gen_id in [5, 6]: # Evangelhos/Epístolas
                if best_score > 0.60 or (best_score > 0.45 and margem > 0.10):
                    decisao_id, status = best_idx, "Resgate/Consolo"
            else:
                if best_score > 0.75:
                    decisao_id, status = best_idx, "Padrão Geral"

    rows_to_persist.append({
        'verso_id': row['verso_id'],
        'topico_id': int(decisao_id),
        'p_exaustao': float(p_ex),
        'p_transitoriedade': float(p_tr),
        'p_vazio': float(p_va),
        'p_narrativo': float(p_na),
        'similaridade_final': float(best_score if decisao_id != 3 else p_na),
        'margem_dominancia': float(margem),
        'status_decisao': status,
        'entropia': float(entropia),
        'gap_confianca': float(gap)
    })

# 4. Persistência Final no SQLite
print("💾 Persistindo classificação e métricas no banco de dados...")
df_final = pd.DataFrame(rows_to_persist)

cursor = conn.cursor()

# --- CORREÇÃO: GARANTIR DADOS NA TABELA 'topico' ---
# Usamos INSERT OR IGNORE para cadastrar os eixos de forma segura.
# Se eles já existirem, o banco ignora e não dá erro de chave duplicada.
mapa_eixos = [
    (0, "Exaustão vs. Refrigério"),
    (1, "Transitoriedade vs. Solidez"),
    (2, "Vazio vs. Propósito"),
    (3, "Narrativo/Normativo")
]
cursor.executemany("""
    INSERT OR IGNORE INTO topico (id, antidoto_referencia)
    VALUES (?, ?)
""", mapa_eixos)

# Deleção seletiva para o livro de Jó
cursor.execute("DELETE FROM verso_topico WHERE verso_id IN (SELECT id FROM verso WHERE livro_id = 18)")

# Persistência das classificações
df_final.to_sql('verso_topico', conn, if_exists='append', index=False)

conn.commit()
conn.close()
print(f"✨ Processamento de Jó concluído! {len(df_final)} versículos processados com contexto original ampliado.")

In [ ]:
# Célula 5.1: Classificação por Eixos Existenciais (Versão Jó/PoC com Janelamento no Texto Original)
import sqlite3
import chromadb
import os
import json
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import torch
from google import genai
from google.colab import userdata
from tqdm.notebook import tqdm

# --- 1. CONFIGURAÇÃO DOS CAMINHOS E CONEXÕES ---
base_dir = "/content/drive/MyDrive/pln/hybrid-rag-poc/data"
chroma_path = os.path.join(base_dir, "chroma_db_jo")
sqlite_db_path = os.path.join(base_dir, "base-dados.db") # Ajuste para o nome real do seu arquivo .db se necessário

# Conecta ao ChromaDB
chroma_client = chromadb.PersistentClient(path=chroma_path)
collection = chroma_client.get_collection(name="comentario_moody_jo")

# Conecta ao SQLite existente
conn = sqlite3.connect(sqlite_db_path)
cursor = conn.cursor()

# --- 2. CARREGAR MODELO DE EMBEDDINGS (BERTimbau) ---
print("🔄 Carregando BERTimbau para gerar embeddings das consultas...")
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def gerar_embedding(texto):
    inputs = tokenizer(texto, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1)[0].cpu().numpy().tolist()

# --- 3. IDENTIFICAÇÃO DOS CASOS CRÍTICOS (FILTRAGEM CIRÚRGICA) ---
query_criticos = """
    SELECT v.id as verso_id, v.numero_capitulo, v.numero_verso, v.texto,
           vt.topico_id, vt.entropia, vt.gap_confianca, vt.status_decisao
    FROM verso_topico vt
    JOIN verso v ON v.id = vt.verso_id
    WHERE v.livro_id = 18
      AND (vt.entropia > 0.88 AND vt.gap_confianca < 0.02)
"""
df_criticos = pd.read_sql_query(query_criticos, conn)
print(f"🔍 RAG ativado seletivamente! Encontrados {len(df_criticos)} versículos de alta ambiguidade real em Jó.")

# --- 4. FUNÇÃO DO RETRIEVER DINÂMICO ---
def buscar_contexto_no_chroma(texto_verso):
    q_emb = gerar_embedding(texto_verso)
    resultados = collection.query(
        query_embeddings=[q_emb],
        n_results=2
    )
    docs = resultados['documents'][0]
    metas = resultados['metadatas'][0]

    contexto_formatado = ""
    for doc, meta in zip(docs, metas):
        contexto_formatado += f"- [Capítulo: {meta['capitulo']} | Seção: {meta['nivel_1']}]\n{doc}\n\n"

    return contexto_formatado.strip()

# --- 5. CONFIGURAÇÃO DA API DO GEMINI ---
try:
    api_key = userdata.get('GOOGLE_API_KEY')
    client_gemini = genai.Client(api_key=api_key)
except Exception as e:
    print("⚠️ Certifique-se de configurar a GOOGLE_API_KEY nos segredos do Colab.")

def chamar_gemini_arbitragem(verso_texto, contexto_comentario):
    prompt = f"""
Você é um auditor de PLN especializado em análise teológica e filosófica de textos bíblicos.
Analise o versículo abaixo do livro de Jó à luz do comentário exegético recuperado e classifique-o ESTRITAMENTE em um dos 4 eixos:

0: Exaustão vs. Refrigério (Fadiga existencial, dor física/mental, lamento nas cinzas, busca por descanso)
1: Transitoriedade vs. Solidez (Brevidade da vida, impermanência x Rocha Eterna, firmeza espiritual)
2: Vazio vs. Propósito (Crise de sentido, futilidade, vaidade x Vocação, soberania divina, luz na escuridão)
3: Narrativo/Normativo (Relato factual, diálogos introdutórios, censos, encerramento biográfico sem carga exaustiva)

[CONTEXTO EXEGÉTICO DO COMENTÁRIO MOODY]:
{contexto_comentario}

[VERSÍCULO ALVO]:
"{verso_texto}"

Responda EXCLUSIVAMENTE em formato JSON contendo exatamente estas chaves:
{{
  "topico_id": <inteiro 0, 1, 2 ou 3>,
  "justificativa": "<explicação curta fundamentada no comentário>",
  "status_decisao": "RAG_Resgate"
}}
"""
    try:
        response = client_gemini.models.generate_content(
            model='gemini-3.6-flash',
            contents=prompt,
            config={
                "temperature": 0.1,
                "response_mime_type": "application/json",
            }
        )
        return json.loads(response.text)
    except Exception as e:
        print(f"Erro na chamada do Gemini: {e}")
        return None

# --- 6. EXECUTAR LOOP DE ARBITRAGEM E ARMAZENAGEM ---
novas_decisoes_contador = 0

if len(df_criticos) > 0:
    print("🚀 Executando o comitê de arbitragem seletiva via RAG e gravando na tabela de auditoria...")
    for _, row in tqdm(df_criticos.iterrows(), total=len(df_criticos), desc="Arbitrando Versos"):
        contexto_teologico = buscar_contexto_no_chroma(row['texto'])
        resposta_json = chamar_gemini_arbitragem(row['texto'], contexto_teologico)

        if resposta_json and 'topico_id' in resposta_json:
            novo_topico_id = int(resposta_json['topico_id'])
            justificativa_texto = str(resposta_json['justificativa'])
            status = str(resposta_json['status_decisao'])
            verso_id = int(row['verso_id'])
            topico_anterior_id = int(row['topico_id'])

            # 1. Atualiza a tabela principal de tópicos e o status
            cursor.execute("""
                UPDATE verso_topico
                SET topico_id = ?,
                    status_decisao = ?
                WHERE verso_id = ?
            """, (novo_topico_id, status, verso_id))

            # 2. Insere na tabela de auditoria enxuta respeitando o schema oficial
            cursor.execute("""
                INSERT OR REPLACE INTO rag_auditoria (
                    verso_id, topico_id_anterior, topico_id_novo, justificativa
                ) VALUES (?, ?, ?, ?)
            """, (
                verso_id,
                topico_anterior_id,
                novo_topico_id,
                justificativa_texto
            ))

            novas_decisoes_contador += 1

    conn.commit()
    print(f"✨ Sucesso! {novas_decisoes_contador} versos críticos reclassificados e registrados na auditoria.")
else:
    print("Nenhum verso crítico encontrado para processar com os filtros atuais.")

conn.close()

In [ ]:
# Célula 6: Análise de Sentimento Contextual e Cruzamento Existencial
from pysentimiento import create_analyzer
import pandas as pd
import sqlite3
from tqdm.auto import tqdm

# 1. Inicializar o Analisador
print("🚀 Carregando modelo Transformer para Sentimento (PT-BR)...")
# O analisador 'sentiment' para 'pt' é baseado em BERTimbau, ideal para o TCC
analyzer = create_analyzer(task="sentiment", lang="pt")

# 2. Busca do texto original e dos tópicos
conn = sqlite3.connect(DB_PATH)
df_input = pd.read_sql_query("""
    SELECT v.id as verso_id, v.texto, vt.topico_id
    FROM verso v
    JOIN verso_topico vt ON v.id = vt.verso_id
""", conn)

textos = df_input['texto'].tolist()
verso_ids = df_input['verso_id'].tolist()

# 3. Execução da análise em lotes (Aproveitando a GPU se disponível)
print(f"📊 Analisando carga emocional de {len(textos)} versículos...")
sentimentos = []
batch_size = 64
mapa_num = {'POS': 1, 'NEU': 0, 'NEG': -1}

# O predict em lote é significativamente mais rápido no Colab
for i in tqdm(range(0, len(textos), batch_size)):
    lote = textos[i:i + batch_size]
    ids_lote = verso_ids[i:i + batch_size]
    preds_lote = analyzer.predict(lote)

    for idx, p in enumerate(preds_lote):
        # Capturamos as probabilidades brutas para análises de incerteza se necessário
        sentimentos.append({
            'verso_id': ids_lote[idx],
            'label': p.output,
            'sentimento_num': mapa_num.get(p.output, 0),
            'score_pos': p.probas.get('POS', 0),
            'score_neg': p.probas.get('NEG', 0),
            'score_neu': p.probas.get('NEU', 0)
        })

df_sent = pd.DataFrame(sentimentos)

# 4. Persistência dos Resultados
try:
    cursor = conn.cursor()
    # Limpamos para garantir que a nova classificação da Célula 5 seja a única presente
    cursor.execute("DELETE FROM verso_sentimento")

    # Inserimos os novos resultados (Integridade referencial com 'verso_id')
    df_sent.to_sql('verso_sentimento', conn, if_exists='append', index=False)
    conn.commit()
    print("\n✅ Célula 6 concluída! Sentimentos processados e salvos com sucesso.")

    # 5. RESULTADO FINAL: O DIAGNÓSTICO (PROBLEMA) VS. A CURA (ANTÍDOTO)
    print("\n📈 RESUMO EXECUTIVO: PROBLEMÁTICA (CRISE) VS. ANTÍDOTO (CURA)")

    res_final = pd.read_sql_query("""
        SELECT
            t.antidoto_referencia as Eixo_Filosofico,
            COUNT(*) as Total_Versos,
            SUM(CASE WHEN vs.sentimento_num = 1 THEN 1 ELSE 0 END) as Antidotos_Cura,
            SUM(CASE WHEN vs.sentimento_num = -1 THEN 1 ELSE 0 END) as Problematica_Crise,
            ROUND(AVG(vs.sentimento_num), 3) as Polaridade_Media
        FROM verso_topico vt
        JOIN topico t ON vt.topico_id = t.id
        JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id
        WHERE t.id != 3 -- Foco nos eixos Han, Bauman e Frankl
        GROUP BY t.antidoto_referencia
        ORDER BY Polaridade_Media DESC
    """, conn)

    # Exibe a tabela formatada no Colab
    display(res_final)

except Exception as e:
    print(f"❌ Erro na persistência: {e}")
finally:
    conn.close()